In [23]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings("ignore")

llm = ChatOllama(
    model="mistral:latest",
    temperature=0.7
)

from pydantic import BaseModel
from typing import Literal

class llm_schema(BaseModel):
    movie_summary_flag: Literal["positive", "negative"]

#llm_structured_output = llm.with_structured_output(llm_schema)

### CHAIN WITH Conditional Chains

In [ ]:
# Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ("system",
     """
     You are a movie review evaluator.

     Respond with ONLY one word:
     positive
     or
     negative
     """),
    ("human",
     "Please categorize the movie review as positive or negative: {input}")
])

# ---------------- Pydantic converter ----------------
def to_pydantic(output: str) -> llm_schema:
    output = output.strip().lower()

    sentiment = "positive" if "positive" in output else "negative"

    return llm_schema(movie_summary_flag=sentiment)

to_pydantic_runnable = RunnableLambda(to_pydantic)

def sentiment_chain_fn(data):
    sentiment_result = (
        prompt_template
        | llm
        | StrOutputParser()
        | to_pydantic_runnable
    ).invoke(data)

    return {
        "text": data["input"],
        "sentiment": sentiment_result.movie_summary_flag
    }

sentiment_chain = RunnableLambda(sentiment_chain_fn)

 

#### Conditional Chain 1

In [26]:
# TASK - 1 [Prompt]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a LinkedIn post generator"),
    ("human", "Create a post for the following text for LinkedIn: {text}")])

# TASK - 2 [LLM]

llm = ChatOllama(
    model="mistral:latest",
    temperature=0.7
)

# TASK - 3 [Str Parser]

str_parser = StrOutputParser()

chain_linkedin = linkedin_prompt | llm | str_parser

#### Conditional Chain 2

In [27]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch

In [ ]:
def insta_chain(text:dict):

    text = text["text"]

    # TASK - 1 [Prompt]
    insta_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Instagram post generator"),
    ("human", "Create a post for the following text for Instagram: {text}")])
    
    # TASK - 2 [LLM]
    llm = ChatOllama(
        model="mistral:latest",
        temperature=0.7
    )
    # TASK - 3 [Str Parser]
    str_parser = StrOutputParser()

    chain_insta = insta_prompt | llm | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

#### **Final Orchestration**

In [38]:
conditional_chain = RunnableBranch(
    (lambda x: x["sentiment"] == "positive", chain_linkedin),
    insta_chain_runnable
)

final_orchestrator = sentiment_chain | conditional_chain

In [41]:
result = final_orchestrator.invoke({
    "input": "I disappoint jism movie"
})

In [42]:
result

' Title: Embracing the Unconventional: A Reflection on "I Disappoint Jism"\n\nHello Connections,\n\nI recently had the opportunity to watch the thought-provoking film, "I Disappoint Jism." This Indian independent movie challenges societal norms and delves deep into themes of self-discovery, individuality, and societal expectations.\n\nThe film, while unconventional, offers a unique perspective on our society\'s pressure to conform and the subsequent struggle for personal identity. It serves as a reminder that it\'s okay to be different, to challenge the status quo, and to embrace our authentic selves.\n\nI highly recommend this movie to anyone seeking thought-provoking content that pushes boundaries and encourages self-reflection. Let\'s continue to support independent cinema that challenges us and inspires us to think differently.\n\n#IDisappointJism #IndependentCinema #SelfDiscovery #SocietalNorms #Authenticity\n\nLooking forward to hearing your thoughts on this thought-provoking fil